# KAIROS VERIFIER — Public Replay

**v1.0 · commit `82399d45` · SHA-256 `f23de8df…`**

Run all cells (Runtime → Run all). No setup, no account, ~3 minutes.

This notebook clones the pinned `v1.0` tag only — never `main` — verifies the script's SHA-256 before running it, installs the exact pinned dependency versions, then compares your result to the published reference.

Full technical dossier: https://github.com/KAIROSSYSTEMSCH/KAIROS-VERIFIER

In [ ]:
#@title Step 1 — Fetch pinned v1.0 (not main)
!rm -rf KAIROS-VERIFIER
!git clone --branch v1.0 --depth 1 https://github.com/KAIROSSYSTEMSCH/KAIROS-VERIFIER.git
%cd KAIROS-VERIFIER

In [ ]:
#@title Step 2 — Verify SHA-256 before running anything
import hashlib, sys

EXPECTED = "f23de8df0d216dc6d79d0402103dcb2bb5ed1132eee83f10bf698c2cbae6d75c"
with open("determinism_ladder.py", "rb") as f:
    got = hashlib.sha256(f.read()).hexdigest()

print("expected:", EXPECTED)
print("got     :", got)

if got != EXPECTED:
    print("\n\u2716 REPLAY NOT VERIFIED — script hash does not match v1.0.")
    print("This does not modify the published proof. Stop here and open an issue.")
    sys.exit(1)
else:
    print("\n\u2713 Script authenticated against v1.0.")

In [ ]:
#@title Step 3 — Install exact dependencies (no venv — Colab runtime is already isolated)
!pip install -q -r requirements.txt

In [ ]:
#@title Step 4 — Run the deterministic ladder (constrained envelope)
!OPENBLAS_CORETYPE=Haswell OMP_NUM_THREADS=1 MKL_NUM_THREADS=1 OPENBLAS_NUM_THREADS=1 PYTHONHASHSEED=0 python3 determinism_ladder.py > ladder_constrained.txt
!cat ladder_constrained.txt

In [ ]:
#@title Step 5 — Compare to the published reference
import re

EXPECTED_HASHES = {
    "L0": "1d9e01d16d638900", "L1": "84af2aaa28305585", "L1b": "9984517da6ef41ff",
    "L2": "86ca842440c89f74", "L3": "60da9d0bfeb7bb99", "L3b": "db791086e49a76e2",
    "L4": "0bf4f040e2ee2721", "L4b": "9d40cfbddb0c14ca", "L5": "687b5cae54c0f69f",
    "L5b": "d36851807abf6013",
}

with open("ladder_constrained.txt") as f:
    text = f.read()

rows = re.findall(r"^(L\d+b?)\s+\S.*?\s([0-9a-f]{16})\s", text, re.MULTILINE)
got = {lvl: h for lvl, h in rows}
l6_hashes = re.findall(r"^L6\s+\S.*?\s([0-9a-f]{16})\s", text, re.MULTILINE)

matched, total = 0, 12
print(f"{'LEVEL':<6}{'YOUR RESULT':<20}{'STATUS'}")
for lvl, ref in EXPECTED_HASHES.items():
    ok = got.get(lvl) == ref
    matched += ok
    print(f"{lvl:<6}{got.get(lvl,'—'):<20}{'MATCH' if ok else 'DIVERGE'}")

l6_ref = "e05c0b80de5ce5e4"
for i, h in enumerate(l6_hashes[:2]):
    ok = h == l6_ref
    matched += ok
    print(f"{'L6#'+str(i+1):<6}{h:<20}{'MATCH' if ok else 'DIVERGE'}")

print("\n" + "="*40)
if matched == total:
    print(f"{matched} / {total} MATCH \u2014 YOUR REPLAY")
    print("REPLAY VERIFIED")
else:
    print(f"{matched} / {total} MATCH \u2014 YOUR REPLAY")
    print("REPLAY NOT FULLY VERIFIED")
    print("\nIf you are on macOS / Apple Silicon, this is expected on L3 (and possibly L5, L6)\u2014")
    print("see the technical dossier for the documented platform-specific reference values.")
    print("This does not modify the published proof.")